In [1]:
print('Ritu')

Ritu


In [2]:
"hi".replace('i','')

'h'

<!-- instead of make it 3 step process
1. generate n title 
2. generate key_point for each title
3. geretate detail slide for each slide 


instead of the we should make this only 2 stap process 
1. generate n title 
2. geretate detail slide for each title

this also reduse the token uses -->


In [ ]:
from langchain_ai21.chat_models import ChatAI21
from langchain.output_parsers import PydanticOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage,BaseMessage,HumanMessage,AIMessage,ToolMessage
from langgraph.types import interrupt,Command 
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition
from typing import TypedDict,Annotated, List, Dict, Literal
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv
import json
import os
load_dotenv()
DB_URL = os.getenv("PPT_URL")

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    feedback: str
    topic: str
    action: Literal[
        "continue_slide",
        "update_outline",
        "update_slide",
        "complete",
        ''
    ]
    tool_caller: Literal[
        "generate_outline",
        "generate_slide_detail"
    ]
class OutlineSlide(BaseModel):
    slide_number: int
    slide_title: str
    key_points: List[str]  
    content_type: str
class OutlineOutput(BaseModel):
    title: str
    total_slides: int
    slides: List[OutlineSlide]
class DetailedPoint(BaseModel):
    key_point: str
    example: str
class DetailedSlideOutput(BaseModel):
    slide_number: int
    slide_title: str
    detailed_content: List[DetailedPoint]
model = ChatAI21(model = 'jamba-mini-2-2026-01')
searchTool = TavilySearchResults(max_results=3)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)
outline_parser = PydanticOutputParser(pydantic_object= OutlineOutput)
detailed_parser = PydanticOutputParser(pydantic_object= DetailedSlideOutput)
OUTLINE_SYSTEM_PROMPT = SystemMessage(
    content=f"""You are an expert presentation designer.

{outline_parser.get_format_instructions()} 

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)
DETAIL_SYSTEM_PROMPT = SystemMessage(
    content=f"""
You are an expert content writer for presentations.

Your task: Generate DETAILED, ENGAGING content for a specific slide.

{detailed_parser.get_format_instructions()}

For each key point:
- Provide 2-3 sentences of explanation
- Include relevant examples, statistics, or facts
- Make it clear, concise, and presentation-ready
- Use simple language that's easy to understand
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)
def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    print('inside generate_outline_node')
    messages = state["messages"] + [OUTLINE_SYSTEM_PROMPT]
    result = model_with_tools.invoke(messages)
    output = {
        'messages':[result],
        'current_slide_index':0,
        "tool_caller": "generate_outline",
            } 
    if result.content:
        try:
            json_str = result.content.strip()
            if '```json' in json_str:
                json_str = json_str.replace('```json','').replace('```','')
            output['outline'] = outline_parser.parse(json_str).model_dump()
        except json.JSONDecodeError as e:
            print('generate_outline_node',e)
    return output
def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    detailed_slides = state.get('detailed_slides',[])
    total_slides = len(state.get('outline',{}).get('slides',[]))
    if current_index >= total_slides:
        return {"action": "complete"}
    output = {
        "tool_caller": "generate_slide_detail",
        "current_slide_index":current_index
    }
    if state['action'] == "update_slide":
        feedback = state['feedback']
        last_slide = detailed_slides.pop()
        last_outline = outline['slides'][current_index-1]
        output['feedback'] = ''
        output['action'] = ''
        prompt = HumanMessage(
            content=f"""
You are updating a single slide in a PowerPoint presentation.
Presentation Title:
{state['outline']['title']}

Outline of the slide:
{last_outline}

Current Slide Content:
{last_slide}

User Feedback:
{feedback}
"""
        )
    else:
        current_slide = outline['slides'][int(current_index)]
        prompt = HumanMessage(
            content=f"""Generate detailede content for this slide:
Slide Title: {current_slide['slide_title']}
Slide Number: {current_slide['slide_number']}
Key Points: {', '.join(current_slide['key_points'])}
Content Type: {current_slide['content_type']}

Provide comprehensive, presentation-ready content."""
    )
    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    if isinstance( state['messages'][-1],ToolMessage):
        messages = state['messages'][-2:]+[DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    if result.content:
        try:
            json_str = result.content.strip()
            if '```json' in json_str:
                json_str = json_str.replace('```json','').replace('```','')
            detailed_slides.append(detailed_parser.parse(json_str).model_dump())
            output['detailed_slides'] = detailed_slides 
            output['current_slide_index'] = current_index +1
            
        except json.JSONDecodeError as e:
            print('generate_slide_detail_node inside',e)
    if output['current_slide_index'] == total_slides:
        output["action"] =  "complete"
    output['messages'] = [result]
    print('final output',output)
    return output
def route_after_tools(state: PptState):
    return state["tool_caller"]
def human_decision(state: PptState):
    decision = interrupt({})
    print('inside human_decision')
    print('decision',decision)
    if decision['action'] == "update_outline":
        print('inside human_decision update_outline')

        return {
            'action': "update_outline",
            "messages":[decision['feedback']]
            }
    elif decision['action'] == 'continue_slide':
        print('inside human_decision continue_slide')
        return {'action':'continue_slide'}
    elif decision['action'] == 'update_slide':
        print('inside human_decision update_slide')
        return {'action':'update_slide'}
def route_after_human(state: PptState):
    action = state['action']
    if action == 'update_outline':
        return "generate_outline"
    elif action in ('continue_slide', 'update_slide'):
        return "generate_slide_detail"
    elif action == 'complete':  
        return END
    return END
def build_workflow():
    workflow = StateGraph(PptState)
    workflow.add_node("generate_outline", generate_outline_node)
    workflow.add_node("generate_slide_detail", generate_slide_detail_node)
    workflow.add_node("human_decision", human_decision)
    workflow.add_node("tools", ToolNode(tools))
    workflow.add_edge(START, "generate_outline")
    workflow.add_conditional_edges(
        "generate_outline",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )
    workflow.add_conditional_edges(
        "generate_slide_detail",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )
    workflow.add_conditional_edges(
        "tools",
        route_after_tools,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
        },
    )
    workflow.add_conditional_edges(
        "human_decision",
        route_after_human,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
            END: END,
        },
    )
    return workflow
def create_ckeckpointer_and_graph(db_url: str):
    if not db_url:
        raise ValueError('Database Url environment variable not set')
    connection_kwargs = {
            "autocommit": True,
            "prepare_threshold": 0,
        }
    pool = ConnectionPool(
        conninfo=db_url,
            max_size=20,
            kwargs=connection_kwargs,
    )
    checkpointer = PostgresSaver(pool)
    checkpointer.setup()
    workflow = build_workflow()
    graph = workflow.compile(checkpointer=checkpointer)
    return checkpointer, graph

checkpointer, graph = create_ckeckpointer_and_graph(DB_URL)




In [ ]:

topic = "what is india ai summit 2026"
num_slide = 3
config = {'configurable':{'thread_id':'21-02-26-3'}}
state = {
            "messages": [HumanMessage(content=f"Create a {num_slide}-slide presentation outline on: {topic}")],
            "topic":topic,
            "outline": {},
            "detailed_slides": [],
            "current_slide_index": 0,
            "feedback": "",
            "action": "",
            "tool_caller": "generate_outline",
        }
result = graph.invoke(state,config = config)

inside generate_outline_node


KeyboardInterrupt: 

In [67]:
result

{'messages': [HumanMessage(content='Create a 3-slide presentation outline on: what is india ai summit 2026', additional_kwargs={}, response_metadata={}, id='3aa075ed-c0a1-49ab-9ee8-42539e9e067b'),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c7eda-d41b-7b62-90d9-bd4687fdbfc0-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026 official details, agenda, speakers, and purpose'}, 'id': 'chatcmpl-tool-8e0a49170cdaa9ab', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='[{"title": "[PDF] INDIA AI IMPACT SUMMIT 2026 - AI for Social Good", "url": "https://www.povertyactionlab.org/sites/default/files/AI%20Summit%20Agenda%20and%20Speakers_0.pdf", "content": "\u200b INDIA AI IMPACT SUMMIT 2026 AI for Social Good: Impact that Works February 17, 2026 | Bharat Mandapam, New Delhi AGENDA 9:30am–9:35am Welcome 9:35am–10:20am From Algorithms to Outcomes: Building AI that Works for People Opening remarks

In [69]:
state = Command(resume={
    "action":'continue_slide'
    
})
result1 = graph.invoke(state,config = config)
result1

inside human_decision
decision {'action': 'continue_slide'}
inside human_decision continue_slide
inside generate_slide_detail_node
detailed_slides [{'slide_number': 1, 'slide_title': 'Introduction to India AI Summit 2026', 'detailed_content': [{'key_point': 'Official Dates and Venues', 'example': 'The India AI Summit 2026 will take place from February 16–20, 2026, across multiple venues in New Delhi. This multi-venue format allows for broader participation and specialized sessions across different AI domains.'}, {'key_point': 'Hosting Organisations and Partners', 'example': 'The event is officially hosted by the India AI Mission under the Ministry of Electronics and Information Technology (MeitY). It is supported by global partners including Google.org and J-PAL, which bring international expertise and resources to amplify the summit’s impact.'}, {'key_point': 'Purpose and Strategic Focus', 'example': 'The summit’s core objective is to position India as a global AI leader by showcasing

{'messages': [HumanMessage(content='Create a 3-slide presentation outline on: what is india ai summit 2026', additional_kwargs={}, response_metadata={}, id='3aa075ed-c0a1-49ab-9ee8-42539e9e067b'),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c7eda-d41b-7b62-90d9-bd4687fdbfc0-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026 official details, agenda, speakers, and purpose'}, 'id': 'chatcmpl-tool-8e0a49170cdaa9ab', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='[{"title": "[PDF] INDIA AI IMPACT SUMMIT 2026 - AI for Social Good", "url": "https://www.povertyactionlab.org/sites/default/files/AI%20Summit%20Agenda%20and%20Speakers_0.pdf", "content": "\u200b INDIA AI IMPACT SUMMIT 2026 AI for Social Good: Impact that Works February 17, 2026 | Bharat Mandapam, New Delhi AGENDA 9:30am–9:35am Welcome 9:35am–10:20am From Algorithms to Outcomes: Building AI that Works for People Opening remarks

In [71]:
state = Command(resume={
    "action":'continue_slide'
    
})
result2 = graph.invoke(state,config = config)
result2

inside human_decision
decision {'action': 'continue_slide'}
inside human_decision continue_slide
inside generate_slide_detail_node
detailed_slides [{'slide_number': 1, 'slide_title': 'Introduction to India AI Summit 2026', 'detailed_content': [{'key_point': 'Official Dates and Venues', 'example': 'The India AI Summit 2026 will take place from February 16–20, 2026, across multiple venues in New Delhi. This multi-venue format allows for broader participation and specialized sessions across different AI domains.'}, {'key_point': 'Hosting Organisations and Partners', 'example': 'The event is officially hosted by the India AI Mission under the Ministry of Electronics and Information Technology (MeitY). It is supported by global partners including Google.org and J-PAL, which bring international expertise and resources to amplify the summit’s impact.'}, {'key_point': 'Purpose and Strategic Focus', 'example': 'The summit’s core objective is to position India as a global AI leader by showcasing

{'messages': [HumanMessage(content='Create a 3-slide presentation outline on: what is india ai summit 2026', additional_kwargs={}, response_metadata={}, id='3aa075ed-c0a1-49ab-9ee8-42539e9e067b'),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c7eda-d41b-7b62-90d9-bd4687fdbfc0-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026 official details, agenda, speakers, and purpose'}, 'id': 'chatcmpl-tool-8e0a49170cdaa9ab', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='[{"title": "[PDF] INDIA AI IMPACT SUMMIT 2026 - AI for Social Good", "url": "https://www.povertyactionlab.org/sites/default/files/AI%20Summit%20Agenda%20and%20Speakers_0.pdf", "content": "\u200b INDIA AI IMPACT SUMMIT 2026 AI for Social Good: Impact that Works February 17, 2026 | Bharat Mandapam, New Delhi AGENDA 9:30am–9:35am Welcome 9:35am–10:20am From Algorithms to Outcomes: Building AI that Works for People Opening remarks

In [63]:
state = Command(resume={
    "action":'continue_slide'
    
})
result3 = graph.invoke(state,config = config)
result3

inside generate_slide_detail_node
detailed_slides [{'slide_number': 1, 'slide_title': 'Introduction to India AI Summit 2026', 'detailed_content': [{'key_point': 'Hosted by Government of India', 'example': 'Organized under the IndiaAI Mission, Ministry of Electronics and Information Technology (MeitY), the Summit reflects national commitment to AI innovation and policy development.'}, {'key_point': 'Main Event Dates', 'example': 'Core Summit activities will run from 19–20 February 2026 at Bharat Mandapam, New Delhi, bringing together policymakers, researchers, and industry leaders.'}, {'key_point': 'Research Symposium', 'example': 'A dedicated Research Symposium on AI and its Impact will be held on 18 February 2026 at Bharat Mandapam, featuring IIIT-Hyderabad as the knowledge partner, fostering interdisciplinary dialogue.'}, {'key_point': 'Collaboration with IIIT-Hyderabad', 'example': 'IIIT-Hyderabad’s involvement as knowledge partner underscores academic-government synergy, enhancing 

{'messages': [HumanMessage(content='Create a 3-slide presentation outline on: what is india ai summit 2026', additional_kwargs={}, response_metadata={}, id='4fa2d437-18ef-4d3b-8e2f-96b594e2f54f'),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c7ec8-8a35-7f03-bc0f-2ccb47191d3f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026 official details, agenda, themes, and participants'}, 'id': 'chatcmpl-tool-94ed33723e866be2', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='[{"title": "Research Symposium - India AI Impact Summit 2026", "url": "https://impact.indiaai.gov.in/events/research-symposium", "content": "As part of the India-AI Impact Summit to be held between 19–20 February 2026, New Delhi, the Government of India—under the IndiaAI Mission, Ministry of Electronics and Information Technology—will host a one-day “Research Symposium on AI and its Impact” on 18 February 2026 at Bharat Man

In [65]:
state = Command(resume={
    "action":'continue_slide'
    
})
result4 = graph.invoke(state,config = config)
result4

inside human_decision
decision {'action': 'continue_slide'}
inside human_decision continue_slide
inside generate_slide_detail_node
detailed_slides [{'slide_number': 1, 'slide_title': 'Introduction to India AI Summit 2026', 'detailed_content': [{'key_point': 'Hosted by Government of India', 'example': 'Organized under the IndiaAI Mission, Ministry of Electronics and Information Technology (MeitY), the Summit reflects national commitment to AI innovation and policy development.'}, {'key_point': 'Main Event Dates', 'example': 'Core Summit activities will run from 19–20 February 2026 at Bharat Mandapam, New Delhi, bringing together policymakers, researchers, and industry leaders.'}, {'key_point': 'Research Symposium', 'example': 'A dedicated Research Symposium on AI and its Impact will be held on 18 February 2026 at Bharat Mandapam, featuring IIIT-Hyderabad as the knowledge partner, fostering interdisciplinary dialogue.'}, {'key_point': 'Collaboration with IIIT-Hyderabad', 'example': 'IIIT

{'messages': [HumanMessage(content='Create a 3-slide presentation outline on: what is india ai summit 2026', additional_kwargs={}, response_metadata={}, id='4fa2d437-18ef-4d3b-8e2f-96b594e2f54f'),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c7ec8-8a35-7f03-bc0f-2ccb47191d3f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026 official details, agenda, themes, and participants'}, 'id': 'chatcmpl-tool-94ed33723e866be2', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='[{"title": "Research Symposium - India AI Impact Summit 2026", "url": "https://impact.indiaai.gov.in/events/research-symposium", "content": "As part of the India-AI Impact Summit to be held between 19–20 February 2026, New Delhi, the Government of India—under the IndiaAI Mission, Ministry of Electronics and Information Technology—will host a one-day “Research Symposium on AI and its Impact” on 18 February 2026 at Bharat Man

In [41]:
# !pip show langgraph
# Name: langgraph
# Version: 1.0.5


Name: langgraph
Version: 1.0.5
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License-Expression: MIT
Location: C:\Users\kaushal\Desktop\me\AI-Powered PPT Generator\pptenv\Lib\site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: 


In [13]:
result

{'messages': [HumanMessage(content='Create a 3-slide presentation outline on: what is india ai summit 2026', additional_kwargs={}, response_metadata={}, id='6f1f6970-5ea5-424a-baf2-19412923b287'),
  HumanMessage(content='Create a 3-slide presentation outline on: what is india ai summit 2026', additional_kwargs={}, response_metadata={}, id='6626f1a3-d79a-4179-a3f9-f0608a545a68'),
  AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c7c21-cf97-7cd1-89cb-5e34ff1ee60e-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026'}, 'id': 'chatcmpl-tool-97a1e03b87a76684', 'type': 'tool_call'}], invalid_tool_calls=[]),
  ToolMessage(content='[{"title": "Event | India AI Impact Summit 2026 - World Bank", "url": "https://www.worldbank.org/en/events/2026/02/05/india-ai-impact-summit-2026", "content": "The India AI Impact Summit 2026 brings together governments, international organizations, industry, and civil society to shape how ar

inside generate_slide_detail_node
state content='Create a 3-slide presentation outline on: what is india ai summit 2026' additional_kwargs={} response_metadata={} id='6f1f6970-5ea5-424a-baf2-19412923b287'
state content='Create a 3-slide presentation outline on: what is india ai summit 2026' additional_kwargs={} response_metadata={} id='6626f1a3-d79a-4179-a3f9-f0608a545a68'
state content='' additional_kwargs={} response_metadata={} id='lc_run--019c7c21-cf97-7cd1-89cb-5e34ff1ee60e-0' tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026'}, 'id': 'chatcmpl-tool-97a1e03b87a76684', 'type': 'tool_call'}] invalid_tool_calls=[]
state content='[{"title": "Event | India AI Impact Summit 2026 - World Bank", "url": "https://www.worldbank.org/en/events/2026/02/05/india-ai-impact-summit-2026", "content": "The India AI Impact Summit 2026 brings together governments, international organizations, industry, and civil society to shape how artificial intelligence can d

NotImplementedError: Message as a sequence must be (role string, template)

In [39]:
a = graph.get_state(config = {'configurable':{'thread_id':'20-02-26-1'}}).values
isinstance( a['messages'][-1],ToolMessage)

ValueError: Message dict must contain 'role' and 'content' keys, got {'slide_number': 1, 'slide_title': 'Purpose and Key Themes', 'detailed_content': [{'key_point': 'Focus on AI for inclusive growth', 'example': 'AI has the potential to transform public services and drive equitable development, but only if designed with inclusivity in mind. The India AI Impact Summit 2026 exemplifies this by bringing together stakeholders to shape AI solutions that uplift marginalized communities—such as improving health systems and economic opportunities for low-income populations.'}, {'key_point': 'Collaboration across sectors', 'example': 'Effective AI integration requires multi-stakeholder engagement. Governments, industry leaders, academia, and civil society are partnering to co-create solutions—like the World Bank’s Summit, which integrates policy coordination, infrastructure access, and real-world applications to ensure AI benefits all.'}, {'key_point': 'Core pillars: policy, infrastructure, app

In [25]:
a['messages'][-2:]

[AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c7e51-831c-71d2-8e4f-0c4c5529334f-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'AI for inclusive growth public services sustainable development collaboration governments industry academia civil society policy coordination access AI infrastructure real-world applications digital divides 2.2 billion offline lack access data devices statistics'}, 'id': 'chatcmpl-tool-98f04f71ebdaff84', 'type': 'tool_call'}], invalid_tool_calls=[]),
 ToolMessage(content='[{"title": "Event | India AI Impact Summit 2026 - World Bank", "url": "https://www.worldbank.org/en/events/2026/02/05/india-ai-impact-summit-2026", "content": "Artificial intelligence is often described as one of the most significant technological breakthroughs in decades.Yet as the world moves rapidly toward an AI-driven future, a critical question remains: will AI narrow global divides, or widen them?Today,2.2 billion people rema

In [9]:
# model.invoke("what is india ai summit 2026")
# AIMessage(content='As of now (2024), **there is no officially announced “India AI Summit 2026”**.\n\nIt’s likely that you’re referring to one of the following:\n\n### 1. **India AI Summit (2023–2024)**\nIndia has already held multiple AI summits under the banner of **India AI Summit** organized by the **Ministry of Electronics and Information Technology (MeitY)** and other government bodies. For example:\n- **India AI Summit 2023**: Held in Bengaluru in November 2023, featuring global AI leaders, startups, academia, and policymakers.\n- **India AI Summit 2024**: Expected to be held in late 2024 or early 2025 (as of mid-2024, details are still emerging).\n\nThese summits are annual events aimed at accelerating AI adoption, promoting innovation, and aligning India’s AI strategy with global trends.\n\n### 2. **Possible Future Event (2026)**\nWhile **no official announcement** exists yet for a “2026” edition, it is **very plausible** that:\n- The next India AI Summit after 2024 will be scheduled for **2025** or **2026**.\n- Government agencies may announce a 2026 summit in late 2024 or 2025 as part of their long-term planning.\n\n### 3. **Other Related Events**\nYou might also be thinking of:\n- **World AI Show India** (organized by Trescon)\n- **AI Summit India by NASSCOM**\n- **Google AI India Summit** or **Microsoft AI India Events**\n\nThese are private or industry-led events that sometimes co-occur with government initiatives.\n\n---\n\n### ✅ Recommendation:\nTo stay updated:\n1. Visit the official website: [https://www.meity.gov.in](https://www.meity.gov.in)\n2. Follow MeitY on Twitter/X: [@MeitY](https://twitter.com/MeitY)\n3. Subscribe to newsletters from NASSCOM or AI industry associations.\n\n> **Bottom line**: There is no confirmed “India AI Summit 20', additional_kwargs={}, response_metadata={}, id='lc_run--019c720d-b325-7070-a39f-4d51d01ef286-0', tool_calls=[], invalid_tool_calls=[])


# result1 = model_with_tools.invoke("what is india ai summit 2026")
# AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c720e-d8e8-7f31-ab30-a77c8f8ecab4-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026'}, 'id': 'chatcmpl-tool-9db5f2707bb7d80e', 'type': 'tool_call'}], invalid_tool_calls=[])


# result2 =  models_with_outline.invoke("what is india ai summit 2026")

# result2 is None


# now understand the problem I'm usnig AI21 lap model which is train in till 2024 and it runing 2026
# when i ask for the only model it says it dont know
# when i use the mode with the tool then model decide to call tool
# when i use model with tool and stricture output it return None because first model decide to call tool after that structude output try to convert this to the pydentice object which result None


# now to solve this problem tool call with structure output with out multiple llm call because llm call are costly


In [ ]:
# result1 = model_with_tools.invoke("what is india ai summit 2026")

In [ ]:
# result1

AIMessage(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019c720e-d8e8-7f31-ab30-a77c8f8ecab4-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India AI Summit 2026'}, 'id': 'chatcmpl-tool-9db5f2707bb7d80e', 'type': 'tool_call'}], invalid_tool_calls=[])

In [ ]:
# result2 =  models_with_outline.invoke("what is india ai summit 2026")
# print(result2)

None


In [10]:
# topic = "what is india ai summit 2026"
# num_slide = 3
# config = {'configurable':{'thread_id':'19-02-26-1'}}
# state = {
#             "messages": [HumanMessage(content=f"Create a {num_slide}-slide presentation outline on: {topic}")],
#             "topic":topic,
#             "outline": {},
#             "detailed_slides": [],
#             "current_slide_index": 0,
#             "feedback": "",
#             "action": "",
#             "tool_caller": "generate_outline",
#         }
# result = graph.invoke(state,config = config)

In [ ]:
# l = [1,[2,3,[4,5,[6,7],[8,9],[10,11]]]]

# def flate(lis):
#     return[
#         item
#         for ele in lis
#         for item in ( flate(ele)  if isinstance(ele,list) else [ele])
#     ]

# flate(l)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]